<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_49_ansible_for_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🛠️ **Module 49 — Ansible for ML Provisioning (configuring the boxes Terraform creates)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 49 — Ansible for ML Provisioning

> Terraform (M48) creates the **boxes**. Helm (M47) configures things **inside** Kubernetes. **Ansible** configures the **boxes themselves** — installing CUDA drivers on bare-metal training rigs, hardening SSH, mounting NFS for shared model caches, rolling NVIDIA driver updates across 200 servers without dropping the cluster. It's **agentless** (just SSH), **idempotent** (run it 100 times — same result), and shines exactly where your infrastructure isn't fully containerised.
>
> Where you'll meet it in ML: on-prem GPU servers, edge devices, training-cluster bootstrap, Kubernetes nodes that need a custom kernel module, and any "we have 30 Linux boxes" situation.

> 🟡 We won't SSH from this notebook (no remote hosts), but every snippet is production-shaped YAML you can drop into a real `playbooks/` folder.

### What you'll cover
1. Why Ansible — and where it fits next to Terraform / Helm
2. The mental model: inventory · playbook · task · module · role
3. **Inventory** — describing your hosts
4. **Modules + tasks** — the unit of work
5. **Idempotency** — the most important property
6. **Variables, facts, templates** (`jinja2`)
7. **Roles** — reusable bundles (the Ansible Galaxy ecosystem)
8. **Handlers** — restart-on-change
9. ML-specific play: install NVIDIA driver + CUDA + Docker + NVIDIA Container Toolkit
10. Production: vault, dynamic inventory, AWX/Tower, drift, anti-patterns


## 1 · Where Ansible fits

| Tool | Role | Stage |
|---|---|---|
| **Terraform / OpenTofu** (M48) | declare infrastructure (VPC, EKS, GPU instance) | "Day 0" — provision |
| **Ansible** | configure the OS / apps **on** that infrastructure | "Day 1" — bootstrap |
| **Helm** (M47) | configure things inside Kubernetes | "Day 1+" — deploy |
| **Argo CD / Flux** | continuously reconcile k8s state from Git | "Day 2+" — operate |

A typical ML platform pipeline:

```
PR merged
  └─► Terraform creates a 16-node DGX cluster
        └─► Ansible installs CUDA 12.4, NVIDIA driver, Docker, NCCL,
            mounts /shared from NFS, hardens SSH, configures auditd
              └─► kubeadm/k3s join → cluster ready
                    └─► Argo CD applies ML-platform Helm releases
```

**You can skip Ansible** if every host is fully cloud-managed (EKS managed node groups bake their own AMIs). You **need Ansible** the moment you have *bare-metal*, *edge devices*, or *custom AMIs*.

## 2 · Mental model

| Concept | What it is |
|---|---|
| **Inventory** | a list of hosts + groups |
| **Playbook** | a YAML file: "for these hosts, run these tasks" |
| **Task** | one call to a **module** (idempotent action) |
| **Module** | a unit like `apt`, `copy`, `systemd`, `nvidia.nvidia_driver` |
| **Role** | a reusable bundle of tasks + files + templates + defaults |
| **Handler** | a task that only runs when **notified** (e.g. "restart nginx") |
| **Vault** | encrypted secrets file Ansible decrypts at runtime |

Two equally valid project layouts:
```
project/                              project/
├── inventory.ini                     ├── inventories/{prod,staging}/hosts.yml
├── site.yml                          ├── group_vars/all.yml
├── group_vars/all.yml                ├── roles/{cuda,docker,k3s}/...
└── roles/cuda/...                    └── playbooks/{site.yml,upgrade-driver.yml}
```


## 3 · Inventory

In [ ]:
inventory = '''[gpu_workers]
gpu-01 ansible_host=10.0.0.11
gpu-02 ansible_host=10.0.0.12
gpu-03 ansible_host=10.0.0.13

[control_plane]
cp-01 ansible_host=10.0.0.5

[k8s_cluster:children]
gpu_workers
control_plane

[gpu_workers:vars]
ansible_user=ubuntu
ansible_ssh_private_key_file=~/.ssh/ml_cluster.pem
nvidia_driver_branch=550
'''
print(inventory)

**INI format** is fine for small inventories; **YAML** scales better. **Dynamic inventory** plugins query AWS/GCP/Azure at runtime so you don't have to hand-maintain IPs.

## 4 · A first task — install Docker on every GPU worker

In [ ]:
playbook = '''---
- name: Bootstrap GPU workers
  hosts: gpu_workers
  become: true             # sudo
  gather_facts: true       # collect distro / kernel / cpu info first
  tasks:

    - name: Install base packages
      ansible.builtin.apt:
        name:
          - curl
          - ca-certificates
          - gnupg
        state: present
        update_cache: yes

    - name: Add Docker GPG key
      ansible.builtin.apt_key:
        url: https://download.docker.com/linux/ubuntu/gpg
        state: present

    - name: Add Docker apt repo
      ansible.builtin.apt_repository:
        repo: "deb https://download.docker.com/linux/ubuntu {{ ansible_distribution_release }} stable"
        state: present

    - name: Install Docker engine
      ansible.builtin.apt:
        name: ["docker-ce", "docker-ce-cli", "containerd.io"]
        state: present

    - name: Ensure docker is running
      ansible.builtin.systemd:
        name: docker
        state: started
        enabled: yes
'''
print(playbook[:500], "...")

**Reading guide.**
- `hosts: gpu_workers` — pulled from inventory.
- `become: true` — escalate to root via sudo.
- Each `task` calls **one module**. Module name → namespace + name (`ansible.builtin.apt`).
- The whole playbook is **declarative** — re-running it on already-installed hosts is a no-op (next section).

## 5 · Idempotency — Ansible's killer feature

Every well-written Ansible module is **idempotent**: it inspects current state and only acts if it differs from desired state.

```
First run on a fresh box:
  TASK [Install Docker engine] ************
  changed: [gpu-01]

Run again, same playbook:
  TASK [Install Docker engine] ************
  ok: [gpu-01]              ← no-op; Ansible skipped it
```

This is why Ansible is safe to run on a schedule — drift correction. Two big tools you'll use:

- **`--check`** dry-run: tells you what *would* change without doing it.
- **`--diff`** shows file diffs for `template` / `copy` tasks.
- **`changed_when: false`** for `command` modules whose return code doesn't reflect "changed".

**The golden rule.** Avoid raw `shell: ...` and `command: ...` unless absolutely necessary. They're the easiest way to break idempotency.

## 6 · Variables, facts, templates

In [ ]:
template = '''# /etc/nvidia-container-runtime/config.toml ({{ ansible_managed }})
[nvidia-container-cli]
no-cgroups = false
load-kmods = true
ldconfig = "@/sbin/ldconfig.real"

[nvidia-container-runtime]
runtimes = {{ runtimes | to_json }}

# host-specific:
hostname = "{{ inventory_hostname }}"
gpu_count = {{ ansible_facts.devices.nvidia | length }}
'''
print(template)

- `{{ var }}` — Jinja2 substitution.
- `ansible_facts.*` — facts collected on each host (distro, kernel, CPU, GPUs).
- Filters: `| to_json`, `| default("x")`, `| length`, `| basename`, …

Variable precedence (high → low): CLI `--extra-vars` > task vars > role vars > host_vars > group_vars > defaults. **Memorise this** — it's the source of half of all Ansible debugging.

## 7 · Roles — the unit of reuse

In [ ]:
role_layout = '''roles/cuda/
├── defaults/main.yml      # default variable values (lowest precedence)
├── vars/main.yml          # role-level vars (higher precedence)
├── tasks/main.yml         # the entry-point task list
├── handlers/main.yml      # restart-on-change handlers
├── templates/             # jinja2 files
├── files/                 # static files to copy
└── meta/main.yml          # dependencies on other roles, license
'''
print(role_layout)

**Ansible Galaxy** (`ansible-galaxy install nvidia.nvidia_driver`) is the package manager for community roles + collections. For ML there are tested roles for: NVIDIA drivers, NCCL, Docker / containerd, k3s / kubeadm, NFS / Lustre clients, Slurm, JupyterHub, MinIO. Use them; don't reinvent.

**Collections** are the modern packaging unit: a bundle of roles + modules + plugins published to Galaxy, e.g. `community.general`, `kubernetes.core`, `nvidia.nvidia_driver`.

## 8 · Handlers — restart-on-change

In [ ]:
handler_pattern = '''---
- hosts: gpu_workers
  become: true
  tasks:

    - name: Render NVIDIA container runtime config
      ansible.builtin.template:
        src: nvidia-container-runtime.toml.j2
        dest: /etc/nvidia-container-runtime/config.toml
        owner: root
        mode: "0644"
      notify: restart docker         # only fires on change

  handlers:
    - name: restart docker
      ansible.builtin.systemd:
        name: docker
        state: restarted
'''
print(handler_pattern)

Handlers run **once at the end of a play**, only if **notified** by a changed task. This is how you avoid restart storms — bundle ten config edits, restart the daemon once.

## 9 · An ML-real play: NVIDIA driver + CUDA + nvidia-container-toolkit

In [ ]:
ml_play = '''---
- name: Provision a GPU worker
  hosts: gpu_workers
  become: true
  vars:
    cuda_version: "12.4"
    nvidia_branch: "550"

  pre_tasks:
    - name: Disable nouveau (open-source nvidia driver)
      ansible.builtin.copy:
        dest: /etc/modprobe.d/blacklist-nouveau.conf
        content: |
          blacklist nouveau
          options nouveau modeset=0
      register: nouveau

    - name: Rebuild initramfs if changed
      ansible.builtin.command: update-initramfs -u
      when: nouveau.changed

  roles:
    - role: docker
    - role: nvidia.nvidia_driver       # community role — handles dkms + reboot
      vars: { nvidia_driver_branch: "{{ nvidia_branch }}" }

  tasks:
    - name: Add NVIDIA container toolkit repo
      ansible.builtin.shell: |
        curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey \
            | gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
        curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list \
            | sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' \
            | tee /etc/apt/sources.list.d/nvidia-container-toolkit.list
      args: { creates: /etc/apt/sources.list.d/nvidia-container-toolkit.list }

    - name: Install nvidia-container-toolkit
      ansible.builtin.apt:
        name: nvidia-container-toolkit
        state: present
        update_cache: yes
      notify: restart docker

    - name: Configure docker for nvidia runtime
      ansible.builtin.command: nvidia-ctk runtime configure --runtime=docker
      args: { creates: /etc/docker/daemon.json }
      notify: restart docker

    - name: Sanity check — nvidia-smi inside docker
      ansible.builtin.command: docker run --rm --gpus all nvidia/cuda:12.4.0-base-ubuntu22.04 nvidia-smi
      register: smi
      changed_when: false

    - debug: msg="{{ smi.stdout_lines[:3] }}"

  handlers:
    - name: restart docker
      ansible.builtin.systemd: { name: docker, state: restarted }
'''
print(ml_play[:600], "...")

This is the canonical "make a Linux box ready for GPU workloads" play. Run it across 200 hosts and you have a homogeneous training cluster.

## 10 · Production basics

| Topic | What to do |
|---|---|
| **`ansible-vault`** | encrypt secret files: `ansible-vault encrypt group_vars/prod/secrets.yml` |
| **Dynamic inventory** | `aws_ec2`, `gcp_compute`, `kubernetes.core.k8s` plugins query the cloud at runtime |
| **AWX / Ansible Tower / Semaphore** | UI + RBAC + scheduled runs + audit log |
| **CI runs** | run `ansible-lint` + `--check` mode in PRs |
| **Mitogen plugin** | 2-7× speedup on large host counts (caches SSH, reuses interpreters) |
| **Strategy** | `strategy: free` to let hosts run at their own pace; `linear` (default) waits for the slowest |
| **Diff & report** | every prod run → posted to Slack / archived for compliance |

### Anti-patterns
- ❌ Long `shell:` blocks — break idempotency and `--check` mode.
- ❌ Hard-coding host IPs in playbooks. Use inventory + `group_vars`.
- ❌ Mixing **provisioning** (use Terraform) and **configuration** (use Ansible). Pick the right tool per phase.
- ❌ Running `ansible all -a 'sudo systemctl restart docker'` from a laptop. Treat playbooks like code: PR'd, reviewed, tagged, run from CI.
- ❌ Forgetting `--diff` when editing config files — you won't notice the change you broke.

### When you don't need Ansible
- 100% containerised workloads on managed Kubernetes → AMIs are baked, k8s + Helm cover the rest.
- Single fleet of immutable cattle → Packer + auto-replace beats mutate-in-place.


## ✅ Recap
- **Terraform creates boxes; Ansible configures them; Helm deploys to clusters.**
- Inventory + playbook + task + role + handler.
- **Idempotency** is the killer property — modules check current state before acting.
- Variables flow through a precedence chain (CLI > task > role > host_vars > group_vars > defaults).
- **Roles + collections** from Galaxy keep you off the reinvention treadmill.
- Production = AWX/Semaphore + ansible-vault + dynamic inventory + CI runs.

Next: **M50 — Prometheus + Grafana** (observability for the platform you just built).
